# L1 vs L2 Regularization Comparison - Lasso vs Ridge

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

**Steps:**
1. **df.shape**: Check the number of rows (samples) and columns (features)
2. **df.info()**: Get information about data types, non-null counts, and memory usage
3. **df.describe()**: View statistical summary (mean, std, min, max, quartiles) for each feature

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

Missing values can significantly impact model performance. We need to identify and handle them appropriately.

**Steps:**
- Use `df.isnull().sum()` to count missing values in each column
- If missing values exist, we can either:
  - Drop rows with missing values
  - Impute missing values (mean, median, mode, or using ML algorithms)

In this dataset, we can see there are no missing values, so no action is needed.

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

Duplicate rows can bias our model by giving more weight to certain observations. We need to identify and remove them.

**Steps:**
- Use `df.duplicated().sum()` to count duplicate rows
- If duplicates exist, use `df.drop_duplicates()` to remove them

In this dataset, there are no duplicate rows, so no action is needed.

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

We need to separate our data into:
- **X (Features)**: All columns except the target variable (Price)
- **y (Target)**: The variable we want to predict (Price)

**Steps:**
- Use `df.drop("Price", axis=1)` to get features (X)
- Use `df["Price"]` to get target (y)

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

Visualizing the distribution of each feature helps us understand:
- The shape of the data (normal, skewed, etc.)
- Potential outliers
- Whether features need transformation

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

Understanding the relationships between features and the target variable helps in feature selection and understanding the data.

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

Boxplots help identify outliers in each feature. Outliers can significantly affect model performance.

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Standardization)

Feature scaling is crucial for algorithms that use distance-based calculations or gradient descent. It ensures all features contribute equally to the model.

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

Splitting the data into training and testing sets is essential to evaluate model performance on unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## L1 vs L2 Regularization: Mathematical Comparison

### L1 Regularization (Lasso)

**Loss Function:**
$$L(\beta) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} |\beta_j|$$

**Key Characteristics:**
- Uses absolute values of coefficients
- Can shrink coefficients to exactly zero (feature selection)
- Produces sparse models
- Geometric interpretation: diamond-shaped constraint region
- Good when you believe many features are irrelevant

### L2 Regularization (Ridge)

**Loss Function:**
$$L(\beta) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda \sum_{j=1}^{p} \beta_j^2$$

**Key Characteristics:**
- Uses squared values of coefficients
- Shrinks coefficients toward zero but never to exactly zero
- Keeps all features in the model
- Geometric interpretation: circular constraint region
- Good when you believe all features are relevant
- Better for handling multicollinearity

### Key Differences

| Aspect | L1 (Lasso) | L2 (Ridge) |
|--------|-----------|-----------|
| Penalty | |βⱼ| | βⱼ² |
| Feature Selection | Yes (can set to 0) | No (never exactly 0) |
| Sparsity | Produces sparse models | Keeps all features |
| Multicollinearity | Picks one feature | Distributes weight |
| Solution | Non-differentiable at 0 | Differentiable everywhere |
| Use Case | Many irrelevant features | All features relevant |

In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train Both Models with Same Alpha

In [ ]:
alpha = 1.0

# Lasso (L1)
lasso = Lasso(alpha=alpha)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

# Ridge (L2)
ridge = Ridge(alpha=alpha)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

## Compare Performance Metrics

In [ ]:
# Lasso metrics
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
rmse_lasso = mse_lasso**0.5
r2_lasso = r2_score(y_test, y_pred_lasso)

# Ridge metrics
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
rmse_ridge = mse_ridge**0.5
r2_ridge = r2_score(y_test, y_pred_ridge)

print("Lasso (L1) Performance:")
print(f"MAE: {mae_lasso:.4f}")
print(f"RMSE: {rmse_lasso:.4f}")
print(f"R²: {r2_lasso:.4f}")
print(f"Non-zero coefficients: {np.sum(lasso.coef_ != 0)}")
print()
print("Ridge (L2) Performance:")
print(f"MAE: {mae_ridge:.4f}")
print(f"RMSE: {rmse_ridge:.4f}")
print(f"R²: {r2_ridge:.4f}")
print(f"Non-zero coefficients: {np.sum(ridge.coef_ != 0)}")

## Compare Coefficients

In [ ]:
coef_comparison = pd.DataFrame({
    "Feature": X_train.columns,
    "Lasso (L1)": lasso.coef_,
    "Ridge (L2)": ridge.coef_
})

coef_comparison["Difference"] = abs(coef_comparison["Lasso (L1)"] - coef_comparison["Ridge (L2)"])
coef_comparison = coef_comparison.sort_values("Difference", ascending=False)
print(coef_comparison)

## Visualize Coefficient Comparison

In [ ]:
plt.figure(figsize=(12, 6))

x = np.arange(len(X_train.columns))
width = 0.35

plt.bar(x - width/2, lasso.coef_, width, label='Lasso (L1)', alpha=0.8)
plt.bar(x + width/2, ridge.coef_, width, label='Ridge (L2)', alpha=0.8)

plt.xlabel('Features')
plt.ylabel('Coefficient Value')
plt.title('Lasso vs Ridge Coefficient Comparison')
plt.xticks(x, X_train.columns, rotation=45, ha='right')
plt.legend()
plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## Test Different Alpha Values

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

results = []

for alpha in alphas:
    # Lasso
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train, y_train)
    y_pred_lasso = lasso.predict(X_test)
    r2_lasso = r2_score(y_test, y_pred_lasso)
    
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    y_pred_ridge = ridge.predict(X_test)
    r2_ridge = r2_score(y_test, y_pred_ridge)
    
    results.append({
        'Alpha': alpha,
        'Lasso R²': r2_lasso,
        'Ridge R²': r2_ridge,
        'Lasso Non-zero Coefs': np.sum(lasso.coef_ != 0),
        'Ridge Non-zero Coefs': np.sum(ridge.coef_ != 0)
    })

results_df = pd.DataFrame(results)
print(results_df)

## Visualize Performance Across Alpha Values

In [ ]:
plt.figure(figsize=(12, 5))

# R² comparison
plt.subplot(1, 2, 1)
plt.plot(results_df['Alpha'], results_df['Lasso R²'], 'o-', label='Lasso (L1)', marker='o')
plt.plot(results_df['Alpha'], results_df['Ridge R²'], 's-', label='Ridge (L2)', marker='s')
plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('R² Score')
plt.title('R² Score vs Alpha')
plt.legend()
plt.grid(True, alpha=0.3)

# Non-zero coefficients comparison
plt.subplot(1, 2, 2)
plt.plot(results_df['Alpha'], results_df['Lasso Non-zero Coefs'], 'o-', label='Lasso (L1)', marker='o')
plt.plot(results_df['Alpha'], results_df['Ridge Non-zero Coefs'], 's-', label='Ridge (L2)', marker='s')
plt.xscale('log')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Number of Non-zero Coefficients')
plt.title('Feature Selection vs Alpha')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary: When to Use L1 vs L2

**Use Lasso (L1) when:**
- You have many features and believe most are irrelevant
- You want automatic feature selection
- Interpretability is important (fewer features)
- You need a sparse model

**Use Ridge (L2) when:**
- You believe all features are relevant
- You have multicollinearity in your data
- You want to keep all features in the model
- You need stable coefficient estimates

**Use Elastic Net (combination of L1 and L2) when:**
- You want the benefits of both L1 and L2
- You have correlated features
- You want some feature selection but not too aggressive
- You need to balance sparsity and stability